# Run ARC Stratified Split Builder

Clones the ARC repository from GitLab, installs dependencies, and runs the stratified split builder on a scenarios JSONL file.

## 1. Runtime Parameters

Edit these values before running the notebook if needed.

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
SCENARIOS_PATH = PROJECT_DIR / "data" / "scenarios_merged.jsonl"  # change to scenarios.jsonl if needed
OUTPUT_DIR = PROJECT_DIR / "data"
TARGET_SFT_SIZE = 750
MIN_PER_CATEGORY = 20
SEED = 42
FORCE_RECLONE = False


## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y git git-lfs wget > /dev/null
!git lfs install


## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")


## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


## 5. Preflight Check

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.sft_dataset import read_scenarios; print('python ok')",
    ],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("--- STDOUT ---")
print(preflight.stdout)
print("--- STDERR ---")
print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")


## 6. Run the Split Builder

In [ ]:
cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.sft_dataset",
    str(SCENARIOS_PATH),
    "--output-dir",
    str(OUTPUT_DIR),
    "--target-sft-size",
    str(TARGET_SFT_SIZE),
    "--min-per-category",
    str(MIN_PER_CATEGORY),
    "--seed",
    str(SEED),
]

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Split builder failed with exit code {returncode}")


## 7. Inspect Output

In [ ]:
for name in ["sft_train.jsonl", "grpo_train.jsonl", "val.jsonl", "test.jsonl"]:
    path = OUTPUT_DIR / name
    print(f"{name}: {path.exists()} {path.stat().st_size / 1024:.1f} KiB" if path.exists() else f"{name}: missing")
    if path.exists():
        with path.open("r", encoding="utf-8") as handle:
            first_line = handle.readline().strip()
        if first_line:
            print(first_line[:2000])
